# CSE 151B - GRPO Fine-Tuning + Inference (targeting ~80%)

**Full pipeline in one notebook:**

1. Install deps
2. Config
3. Load & format dataset
4. Reward functions
5. Load model with QLoRA
6. GRPO training
7. Save & merge
8. Inference on public set (eval)
9. Retry wrong answers
10. Score summary
11. Generate private set submission

## 1. Install Dependencies

Run once, then comment out. Restart kernel after.

In [ ]:
import sys
!{sys.executable} -m pip install -q \\
    'transformers>=4.51' \\
    'trl==0.17.0' \\
    'peft>=0.11' \\
    'accelerate>=0.27' \\
    'bitsandbytes>=0.43' \\
    datasets requests sympy numpy tqdm \\
    'antlr4-python3-runtime==4.11.1'
print('Done. Restart kernel now.')

## 2. Imports & Configuration

In [ ]:
import json, re, sys, os
from pathlib import Path
from typing import Optional
import torch

print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Paths
BASE_MODEL     = 'Qwen/Qwen3-4B'
PUBLIC_DATA    = 'data/public.jsonl'
PRIVATE_DATA   = 'data/private.jsonl'
OUTPUT_DIR     = 'grpo_output/qwen3-4b-math'
MERGED_DIR     = 'merged_model/qwen3-4b-grpo'
PUBLIC_RESULT  = 'results/grpo_public.jsonl'
PRIVATE_RESULT = 'results/grpo_private.jsonl'

# LoRA
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                  'gate_proj', 'up_proj', 'down_proj']

# GRPO - tuned for A100 40GB
MAX_PROMPT_LEN     = 512
MAX_COMPLETION_LEN = 1024
NUM_GENERATIONS    = 8      # 8 rollouts = strong signal; drop to 4 if OOM
BATCH_SIZE         = 2      # drop to 1 if OOM
GRAD_ACCUM         = 4
NUM_EPOCHS         = 3
LR                 = 5e-6
KL_COEFF           = 0.01

# Inference (after training)
MAX_TOKENS            = 32768
THINKING_BUDGET       = 24000
RETRY_THINKING_BUDGET = 32000
TEMPERATURE           = 0.1

print(f'Base model : {BASE_MODEL}')
print(f'Output dir : {OUTPUT_DIR}')
print(f'Rollouts/Q : {NUM_GENERATIONS}')
print(f'Eff. batch : {BATCH_SIZE * GRAD_ACCUM}')

## 3. System Prompts & Dataset

Improved prompts with self-check, reasoning framing, and stronger MCQ verification.

In [ ]:
SYSTEM_PROMPT_MATH = (
    'You are an expert mathematician. Solve the problem step-by-step.\n\n'

    '=== REASONING APPROACH ===\n'
    'Before writing any equations, spend a moment identifying:\n'
    '  What type of problem is this? (probability, calculus, linear algebra, etc.)\n'
    '  What are the knowns and unknowns?\n'
    '  Is there a simpler equivalent formulation?\n\n'

    '=== FINAL ANSWER FORMAT - THE MOST IMPORTANT RULE ===\n'
    'At the very LAST LINE of your response, place ALL answers inside exactly ONE \\boxed{}.\n'
    '  DO:     \\boxed{380, 315, 13, 310}  (all parts, comma-separated, one box)\n'
    '  DO:     \\boxed{5/8}  (single answer)\n'
    "  DON'T:  box each sub-answer in a separate \\boxed{} throughout the solution\n"
    'Even if you use \\boxed{} for intermediate steps during working, you MUST finish with '
    'a single combined \\boxed{a, b, c} on the very last line - all answers, in the order asked.\n'
    'Never leave \\boxed{} empty.\n\n'

    '=== EXACT FORM RULES ===\n'
    '1. SYMBOLIC OVER NUMERIC: write \\arctan(4.76) not 1.3635\n'
    '2. DECIMAL PRECISION: at minimum 6 significant digits\n'
    '3. PRESERVE STRUCTURE: if the problem writes 2*8*x, write 2*8*x not 16x.\n'
    '4. EXPLICIT MULTIPLICATION: write 3*t*(1-t)^2, not 3t(1-t)^2.\n'
    '5. EXPONENTIALS: write \\exp(0.016*t) or e^{0.016t}.\n'
    '6. FRACTIONS: use exact fractions (5/8) for rational results.\n'
    '7. ORDER: answer multi-part questions in the exact order the problem asks.\n'
    '8. NO ANGLE BRACKETS: do not wrap answers in <> brackets.\n\n'

    '=== SELF-CHECK BEFORE COMMITTING ===\n'
    'Before writing the final \\boxed{}, ask yourself:\n'
    '  Does this answer satisfy all the constraints in the problem?\n'
    '  Did I answer every part that was asked?\n'
    '  Is the form correct (symbolic vs decimal, fraction vs decimal)?\n'
    '  Did I make any sign errors or off-by-one mistakes?\n'
    'If anything seems off, re-examine your setup before committing.\n'
    'Commit to your best answer.'
)

SYSTEM_PROMPT_MCQ = (
    'You are an expert mathematician. '
    'Read the problem and the answer choices carefully, then select the single best answer.\n\n'
    'STEP 1 Solve: Work through the problem step-by-step to derive your answer.\n\n'
    'STEP 2 Match: Compare your result against EVERY option:\n'
    '  a) Check algebraic/symbolic equivalence.\n'
    '  b) Plug in a concrete numeric value for any free variable and evaluate BOTH your '
    'answer and each option - pick the one whose value matches yours.\n'
    '  c) If two options appear numerically equal, prefer the one whose algebraic form '
    'matches your derivation most directly.\n\n'
    'STEP 3 Verify: Before boxing your answer, double-check:\n'
    '  Re-read the question to confirm what is being asked.\n'
    '  Confirm your chosen letter actually corresponds to what you computed.\n'
    '  If uncertain between two options, do a quick numeric sanity check.\n\n'
    'STEP 4 Commit: Trust your derivation.\n\n'
    'You MUST always pick one of the given letters, never say none match. '
    'Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}.'
)


def build_prompt(question: str, options=None) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = '\n'.join(f'{lbl}. {opt.strip()}' for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f'{question}\n\nOptions:\n{opts_text}'
    return SYSTEM_PROMPT_MATH, question


def build_retry_prompt(question: str, options=None) -> tuple[str, str]:
    system, user = build_prompt(question, options)
    prefix = (
        'Take a fresh look at this problem. Do NOT assume your first approach was correct - '
        'try an independent method or re-examine your setup from scratch.\n\n'
    )
    return system, prefix + user


print('Prompts ready.')

## 4. Load & Format Dataset for GRPO

In [ ]:
from datasets import Dataset
import random


def item_to_grpo(item: dict) -> dict:
    system, user = build_prompt(item['question'], item.get('options'))
    answer = item['answer']
    answer_str = json.dumps(answer) if isinstance(answer, list) else str(answer)
    return {
        'prompt': [
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': user},
        ],
        'answer':  answer_str,
        'is_mcq':  bool(item.get('options')),
        'item_id': item['id'],
    }


raw_data = [json.loads(line) for line in open(PUBLIC_DATA)]
random.seed(42)
random.shuffle(raw_data)

split       = int(0.9 * len(raw_data))
train_items = raw_data[:split]
val_items   = raw_data[split:]

train_dataset = Dataset.from_list([item_to_grpo(x) for x in train_items])
val_dataset   = Dataset.from_list([item_to_grpo(x) for x in val_items])

print(f'Train: {len(train_dataset)}  |  Val: {len(val_dataset)}')
s = train_dataset[0]
print(f'  is_mcq : {s["is_mcq"]}')
print(f'  answer : {s["answer"]}')
print(f'  prompt : {s["prompt"][1]["content"][:100]}...')

## 5. Reward Functions

- `correctness_reward` (+1.0): uses judger for free-form, letter match for MCQ
- `format_reward` (+0.2): did the model produce a \\boxed{}? Dense early signal.

In [ ]:
sys.path.insert(0, '.')
from judger import Judger
_judger = Judger(strict_extract=False)


def extract_letter(text: str) -> str:
    m = re.search(r'\\boxed\{\s*([A-Za-z])\s*\.?\}', text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r'\b([A-Z])\b', text.upper())
    return matches[-1] if matches else ''


def has_boxed(text: str) -> bool:
    return bool(re.search(r'\\boxed\{[^}]+\}', text))


def correctness_reward(completions, answer, is_mcq, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        if isinstance(completion, list):
            text = completion[-1].get('content', '') if completion else ''
        elif isinstance(completion, dict):
            text = completion.get('content', '')
        else:
            text = str(completion)
        try:
            if is_mcq:
                correct = (extract_letter(text) == answer.strip().upper())
            else:
                try:
                    gold_list = json.loads(answer)
                    if not isinstance(gold_list, list):
                        gold_list = [gold_list]
                except Exception:
                    gold_list = [answer]
                correct = _judger.auto_judge(text, gold_list,
                                             options=[[]]*len(gold_list))
        except Exception:
            correct = False
        rewards.append(1.0 if correct else 0.0)
    return rewards


def format_reward(completions, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        if isinstance(completion, list):
            text = completion[-1].get('content', '') if completion else ''
        elif isinstance(completion, dict):
            text = completion.get('content', '')
        else:
            text = str(completion)
        rewards.append(0.2 if has_boxed(text) else 0.0)
    return rewards


# Sanity check
dummy = ['The answer is \\boxed{C}', 'I think it might be D maybe', 'Therefore \\boxed{A}']
print('Correctness (gold=C, MCQ):', correctness_reward(dummy, answer='C', is_mcq=True))
print('Format rewards:           ', format_reward(dummy))

## 6. Load Model with QLoRA

QLoRA (4-bit) keeps VRAM usage low while preserving full fine-tuning quality. Takes 3-5 min.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code = True,
    padding_side      = 'left',
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config = bnb_config,
    device_map          = 'auto',
    trust_remote_code   = True,
    torch_dtype         = torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = TARGET_MODULES,
    bias           = 'none',
    task_type      = TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 7. GRPO Training

**If OOM**: reduce `NUM_GENERATIONS` to 4, then `BATCH_SIZE` to 1.

**Watch**: `reward/correctness_reward` in logs - should trend upward each epoch.

In [ ]:
from trl import GRPOTrainer, GRPOConfig

grpo_config = GRPOConfig(
    output_dir                  = OUTPUT_DIR,

    # Rollout
    num_generations             = NUM_GENERATIONS,
    max_prompt_length           = MAX_PROMPT_LEN,
    max_completion_length       = MAX_COMPLETION_LEN,
    temperature                 = 0.8,
    top_p                       = 0.95,

    # Training
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.05,
    bf16                        = True,
    optim                       = 'adamw_torch_fused',

    # GRPO-specific
    beta                        = KL_COEFF,
    use_vllm                    = False,

    # Logging
    logging_steps               = 5,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 2,
    load_best_model_at_end      = False,
    report_to                   = 'none',
    seed                        = 42,

    # Memory
    gradient_checkpointing      = True,
    dataloader_num_workers      = 0,
)

trainer = GRPOTrainer(
    model            = model,
    reward_funcs     = [correctness_reward, format_reward],
    args             = grpo_config,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    processing_class = tokenizer,
)

print('GRPOTrainer ready.')
print(f'Steps per epoch : ~{len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)}')
print(f'Total steps     : ~{len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS}')

In [ ]:
# Run training - ~2 hours on A100 40GB
trainer_stats = trainer.train()
print(f'Training complete. Loss: {trainer_stats.training_loss:.4f}')

## 8. Save Adapter & Merge

In [ ]:
# Save LoRA adapter
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved -> {OUTPUT_DIR}')

# Merge into full model
Path(MERGED_DIR).mkdir(parents=True, exist_ok=True)
merged = model.merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f'Merged model saved -> {MERGED_DIR}')

# Free VRAM
del model, merged, trainer
torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 9. Load Merged Model for Inference

Reloads as fp16 - faster than training mode.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

infer_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR, trust_remote_code=True)
infer_model     = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    device_map        = 'auto',
    torch_dtype       = torch.float16,
    trust_remote_code = True,
)
infer_model.eval()
print(f'Inference model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')


@torch.no_grad()
def generate_response(question: str, options=None, retry=False) -> str:
    if retry:
        system, user = build_retry_prompt(question, options)
        budget = RETRY_THINKING_BUDGET
    else:
        system, user = build_prompt(question, options)
        budget = THINKING_BUDGET

    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': user},
    ]
    input_ids = infer_tokenizer.apply_chat_template(
        messages, return_tensors='pt', add_generation_prompt=True
    ).to(infer_model.device)

    output = infer_model.generate(
        input_ids,
        max_new_tokens = MAX_TOKENS,
        temperature    = TEMPERATURE,
        top_p          = 0.95,
        do_sample      = True,
        pad_token_id   = infer_tokenizer.eos_token_id,
    )
    new_tokens = output[0][input_ids.shape[-1]:]
    return infer_tokenizer.decode(new_tokens, skip_special_tokens=True)


def score_response(item: dict, response: str) -> bool:
    if item.get('options'):
        return extract_letter(response) == item['answer'].strip().upper()
    gold_list = item['answer'] if isinstance(item['answer'], list) else [item['answer']]
    return _judger.auto_judge(response, gold_list, options=[[]]*len(gold_list))


print('generate_response() ready.')

## 10. Evaluate on Public Set

In [ ]:
from tqdm import tqdm

public_data = [json.loads(line) for line in open(PUBLIC_DATA)]
out_path    = Path(PUBLIC_RESULT)
out_path.parent.mkdir(parents=True, exist_ok=True)

done_ids, results = set(), []
if out_path.exists():
    with open(out_path) as f:
        for line in f:
            rec = json.loads(line)
            done_ids.add(rec['id'])
            results.append(rec)
    print(f'Resuming: {len(done_ids)} done, {len(public_data)-len(done_ids)} remaining')
else:
    print(f'Evaluating {len(public_data)} questions...')

with open(out_path, 'a') as f:
    for item in tqdm(public_data, desc='Public eval'):
        if item['id'] in done_ids:
            continue
        try:
            response = generate_response(item['question'], item.get('options'))
            correct  = score_response(item, response)
        except Exception as e:
            print(f'\nERROR id={item["id"]}: {e}')
            response, correct = '', False
        record = {
            'id':       item['id'],
            'is_mcq':   bool(item.get('options')),
            'gold':     item['answer'],
            'response': response,
            'correct':  correct,
        }
        results.append(record)
        f.write(json.dumps(record) + '\n')
        f.flush()

## 11. Retry Wrong + Missing Answers

Retries incorrect answers with rephrased prompt and max thinking budget.

In [ ]:
out_path = Path(PUBLIC_RESULT)

results_by_id = {}
if out_path.exists():
    with open(out_path) as f:
        for line in f:
            rec = json.loads(line)
            results_by_id[rec['id']] = rec

all_ids     = {item['id'] for item in public_data}
missing_ids = all_ids - set(results_by_id.keys())
empty_ids   = {rid for rid, rec in results_by_id.items()
               if not rec.get('response', '').strip()}
wrong_ids   = {rid for rid, rec in results_by_id.items()
               if rec.get('response', '').strip() and not rec.get('correct', True)}

retry_ids = sorted(missing_ids | empty_ids | wrong_ids)
print(f'Missing : {len(missing_ids)}')
print(f'Empty   : {len(empty_ids)}')
print(f'Wrong   : {len(wrong_ids)}')
print(f'Total to retry: {len(retry_ids)}')

data_by_id = {item['id']: item for item in public_data}

def flush_jsonl():
    with open(out_path, 'w') as f:
        for rec in results_by_id.values():
            f.write(json.dumps(rec) + '\n')

n_improved = 0
for fid in tqdm(retry_ids, desc='Retrying'):
    item = data_by_id[fid]
    try:
        response = generate_response(item['question'], item.get('options'), retry=True)
    except Exception as e:
        print(f'\nERROR retry id={fid}: {e}')
        response = results_by_id.get(fid, {}).get('response', '')

    correct = score_response(item, response) if response.strip() else False

    if response.strip():
        old_correct = results_by_id.get(fid, {}).get('correct', False)
        if correct and not old_correct:
            n_improved += 1
            print(f'  FIXED id={fid}')
        results_by_id[fid] = {
            'id':       fid,
            'is_mcq':   bool(item.get('options')),
            'gold':     item['answer'],
            'response': response,
            'correct':  correct,
        }
        flush_jsonl()

print(f'\nDone. {n_improved} answers improved.')

## 12. Score Summary

In [ ]:
results  = [json.loads(line) for line in open(PUBLIC_RESULT)]
mcq_res  = [r for r in results if  r['is_mcq']]
free_res = [r for r in results if not r['is_mcq']]

def acc(subset):
    return sum(r['correct'] for r in subset) / len(subset) * 100 if subset else 0.0

print('=' * 50)
print('GRPO RESULTS')
print('=' * 50)
print(f'  MCQ        : {sum(r["correct"] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)')
print(f'  Free-form  : {sum(r["correct"] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)')
print(f'  Overall    : {sum(r["correct"] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)')
print('=' * 50)

## 13. Generate Private Set Submission

In [ ]:
private_data = [json.loads(line) for line in open(PRIVATE_DATA)]
priv_path    = Path(PRIVATE_RESULT)
priv_path.parent.mkdir(parents=True, exist_ok=True)

done_ids = set()
if priv_path.exists():
    with open(priv_path) as f:
        for line in f:
            done_ids.add(json.loads(line)['id'])
    print(f'Resuming: {len(done_ids)} done, {len(private_data)-len(done_ids)} remaining')
else:
    print(f'Generating {len(private_data)} private responses...')

with open(priv_path, 'a') as f:
    for item in tqdm(private_data, desc='Private inference'):
        if item['id'] in done_ids:
            continue
        try:
            response = generate_response(item['question'], item.get('options'))
        except Exception as e:
            print(f'\nERROR id={item["id"]}: {e}')
            response = ''
        record = {
            'id':       item['id'],
            'is_mcq':   bool(item.get('options')),
            'response': response,
        }
        f.write(json.dumps(record) + '\n')
        f.flush()

print(f'\nPrivate responses saved -> {PRIVATE_RESULT}')
print('Submit this file to the leaderboard.')

---
## Troubleshooting

### OOM during training
Try in order:
```python
NUM_GENERATIONS    = 4    # was 8
BATCH_SIZE         = 1    # was 2
MAX_COMPLETION_LEN = 512  # was 1024
```

### Reward stuck near 0 (all rollouts wrong)
```python
# In grpo_config:
temperature = 1.0   # more diverse rollouts
# In config:
NUM_GENERATIONS = 16  # more chances for at least one correct
```

### Reward collapses after improving
KL too low - model drifting:
```python
KL_COEFF = 0.05   # was 0.01
``
